# SCRIPTY Dataset Processing

This notebook uses the current public-domain source catalog and generated manifest. It does not install packages or require spaCy.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if ROOT.name == 'backend':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
from backend.research.dataset_ingestion import load_sources

sources = load_sources(None, str(ROOT / 'backend/data/source_catalog.json'))
sections = {}
for source in sources:
    sections[source['section']] = sections.get(source['section'], 0) + 1
print({'source_count': len(sources), 'sections': sections})


In [ ]:
from backend.research.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(str(ROOT / 'backend/data/dataset_manifest.jsonl'))
stats = pipeline.stats()
print(json.dumps(stats, indent=2, sort_keys=True))


In [ ]:
results = pipeline.retrieve('colonial india political mystery Delhi', top_k=5, filters={'region': 'south_asia'})
for result in results:
    print(result.source_id, result.passage_id, result.score, result.metadata)
    print(result.text[:300].replace('\n', ' '), '\n')


In [ ]:
from backend.research.neural_reranker import train_from_manifest

report = train_from_manifest(
    manifest_path=str(ROOT / 'backend/data/dataset_manifest.jsonl'),
    model_path=str(ROOT / 'backend/research_output/models/neural_reranker.json'),
    epochs=5,
    max_examples=1000,
)
print(report)
